<a href="https://colab.research.google.com/github/jashvidesai2030/IEMS499Spring/blob/main/ToyExampleBQP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gurobipy

import itertools
import numpy as np
import gurobipy as gp
from gurobipy import GRB

# Data
mu = np.array([3, 4, 5, 2], dtype=float)
b = 7.0

Q = np.array([
    [2, 1, 0, 1],
    [1, 3, 1, 0],
    [0, 1, 4, 1],
    [1, 0, 1, 2]
], dtype=float)

n = len(mu)

# Basic functions
def N_value(x):
    return float((b - mu @ x) ** 2)


def D_value(x):
    return float(x @ Q @ x)


def f_value(x):
    return float((b - mu @ x) / np.sqrt(D_value(x)))


def clean_x(x):
    return tuple(int(v) for v in x)


# Part 1: Enumerate all 16 binary combinations
def brute_force_original():
    results = []

    for bits in itertools.product([0, 1], repeat=n):
        x = np.array(bits, dtype=float)

        # skip zero vector because denominator is zero
        if np.sum(x) == 0:
            continue

        results.append({
            "x": clean_x(x),
            "muTx": float(mu @ x),
            "N": N_value(x),
            "D": D_value(x),
            "f": f_value(x),
            "region": "X+" if mu @ x <= b else "X-"
        })

    results.sort(key=lambda row: row["f"])
    return results


brute_results = brute_force_original()

print("Brute-force enumeration:")
for row in brute_results:
    print(row)

print("\nBrute-force optimum:")
print(brute_results[0])

# Cover and lifted cover generation
def generate_covers_plus():
    """
    Covers for X+:
        mu^T x <= b

    A cover C satisfies:
        sum_{i in C} mu_i > b

    Then:
        sum_{i in C} x_i <= |C| - 1
    """
    covers = []

    for r in range(1, n + 1):
        for C in itertools.combinations(range(n), r):
            if sum(mu[i] for i in C) > b:
                covers.append(C)

    return covers


def generate_covers_minus():
    """
    Covers for X-:
        mu^T x > b

    Equivalently, if all variables in C are zero, then the remaining
    variables cannot exceed b.

    So C is a reverse cover if:
        sum_{i not in C} mu_i <= b

    This implies at least one variable in C must be selected:
        sum_{i in C} x_i >= 1
    """
    covers = []

    all_indices = set(range(n))

    for r in range(1, n + 1):
        for C in itertools.combinations(range(n), r):
            remaining = all_indices.difference(C)

            if sum(mu[i] for i in remaining) <= b:
                covers.append(C)

    return covers

# Build and solve one Dinkelbach subproblem
def solve_subproblem(q, region="plus", add_cuts=True, verbose=False):
    """
    Solves one Dinkelbach subproblem.

    For X+:
        min N(x,y) - q D(x,y)

    For X-:
        max N(x,y) - q D(x,y)

    The model uses y_ij = x_i x_j and BQP constraints.
    """

    model = gp.Model()
    model.Params.OutputFlag = 1 if verbose else 0

    # Binary variables
    x = model.addVars(n, vtype=GRB.BINARY, name="x")

    # Auxiliary variables y_ij for i < j
    y = {}
    for i in range(n):
        for j in range(i + 1, n):
            y[i, j] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}_{j}")

    model.update()

    # Avoid zero denominator
    model.addConstr(gp.quicksum(x[i] for i in range(n)) >= 1)

    # Region constraints
    if region == "plus":
        model.addConstr(gp.quicksum(mu[i] * x[i] for i in range(n)) <= b)
    elif region == "minus":
        # strict inequality mu^T x > b is implemented as >= b + eps
        # for integer data, eps = 1 is fine here
        model.addConstr(gp.quicksum(mu[i] * x[i] for i in range(n)) >= b + 1)
    else:
        raise ValueError("region must be 'plus' or 'minus'")

    # BQP constraints for y_ij = x_i x_j
    for i in range(n):
        for j in range(i + 1, n):
            model.addConstr(y[i, j] <= x[i])
            model.addConstr(y[i, j] <= x[j])
            model.addConstr(y[i, j] >= x[i] + x[j] - 1)

    # Cover and lifted cover cuts
    if add_cuts:
        if region == "plus":
            covers = generate_covers_plus()

            for C in covers:
                # Cover cut:
                # sum_{i in C} x_i <= |C| - 1
                model.addConstr(
                    gp.quicksum(x[i] for i in C) <= len(C) - 1
                )

                # Lifted y-space cover cut:
                # sum_{i<j, i,j in C} y_ij <= ((|C|-2)/2) sum_{i in C} x_i
                if len(C) >= 2:
                    y_pairs = [
                        y[min(i, j), max(i, j)]
                        for i, j in itertools.combinations(C, 2)
                    ]

                    model.addConstr(
                        gp.quicksum(y_pairs)
                        <= ((len(C) - 2) / 2) * gp.quicksum(x[i] for i in C)
                    )

        elif region == "minus":
            covers = generate_covers_minus()

            for C in covers:
                # Reverse cover cut:
                # sum_{i in C} x_i >= 1
                model.addConstr(
                    gp.quicksum(x[i] for i in C) >= 1
                )

    # Build N(x,y)
    N_expr = b ** 2

    for i in range(n):
        N_expr += mu[i] * (mu[i] - 2 * b) * x[i]

    for i in range(n):
        for j in range(i + 1, n):
            N_expr += 2 * mu[i] * mu[j] * y[i, j]

    # Build D(x,y)
    D_expr = 0

    for i in range(n):
        D_expr += Q[i, i] * x[i]

    for i in range(n):
        for j in range(i + 1, n):
            D_expr += 2 * Q[i, j] * y[i, j]

    # Dinkelbach objective
    objective = N_expr - q * D_expr

    if region == "plus":
        model.setObjective(objective, GRB.MINIMIZE)
    else:
        model.setObjective(objective, GRB.MAXIMIZE)

    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Subproblem was not solved to optimality. Status: {model.Status}")

    x_sol = np.array([round(x[i].X) for i in range(n)], dtype=float)

    residual = float(model.ObjVal)
    N_sol = N_value(x_sol)
    D_sol = D_value(x_sol)
    q_new = N_sol / D_sol

    return {
        "x": clean_x(x_sol),
        "x_array": x_sol,
        "N": N_sol,
        "D": D_sol,
        "q_new": q_new,
        "residual": residual,
        "f": f_value(x_sol)
    }

# Dinkelbach algorithm
def dinkelbach(region="plus", q0=0.0, tol=1e-8, max_iter=50, add_cuts=True):
    q = q0
    history = []

    for k in range(max_iter):
        sol = solve_subproblem(
            q=q,
            region=region,
            add_cuts=add_cuts,
            verbose=False
        )

        sol["iteration"] = k
        sol["q"] = float(q)
        history.append(sol)

        if abs(sol["residual"]) <= tol:
            break

        q = sol["q_new"]

    q_star = history[-1]["q_new"]
    x_star = history[-1]["x_array"]

    if region == "plus":
        objective_value = np.sqrt(q_star)
    else:
        objective_value = -np.sqrt(q_star)

    return {
        "region": region,
        "q_star": float(q_star),
        "x_star": clean_x(x_star),
        "objective_value": float(objective_value),
        "history": history
    }

# Run both regions
plus_result = dinkelbach(region="plus", q0=0.0, add_cuts=True)
minus_result = dinkelbach(region="minus", q0=0.0, add_cuts=True)

print("\nDinkelbach X+ result:")
print({
    "q_star": plus_result["q_star"],
    "x_star": plus_result["x_star"],
    "min_X_plus_f": plus_result["objective_value"]
})

print("\nDinkelbach X- result:")
print({
    "q_star": minus_result["q_star"],
    "x_star": minus_result["x_star"],
    "min_X_minus_f": minus_result["objective_value"]
})

global_result = min(
    [plus_result, minus_result],
    key=lambda r: r["objective_value"]
)

print("\nGlobal Dinkelbach result:")
print({
    "region": global_result["region"],
    "x_star": global_result["x_star"],
    "objective_value": global_result["objective_value"]
})

# Print iteration histories cleanly
def print_history(result):
    print(f"\nHistory for {result['region']}:")
    for row in result["history"]:
        print({
            "k": row["iteration"],
            "q_k": row["q"],
            "x": row["x"],
            "N": row["N"],
            "D": row["D"],
            "q_next": row["q_new"],
            "residual": row["residual"],
            "f": row["f"]
        })


print_history(plus_result)
print_history(minus_result)

Brute-force enumeration:
{'x': (1, 1, 1, 1), 'muTx': 14.0, 'N': 49.0, 'D': 19.0, 'f': -1.6059101370939322, 'region': 'X-'}
{'x': (1, 1, 1, 0), 'muTx': 12.0, 'N': 25.0, 'D': 13.0, 'f': -1.386750490563073, 'region': 'X-'}
{'x': (0, 1, 1, 1), 'muTx': 11.0, 'N': 16.0, 'D': 13.0, 'f': -1.1094003924504583, 'region': 'X-'}
{'x': (1, 0, 1, 1), 'muTx': 10.0, 'N': 9.0, 'D': 12.0, 'f': -0.8660254037844387, 'region': 'X-'}
{'x': (0, 1, 1, 0), 'muTx': 9.0, 'N': 4.0, 'D': 9.0, 'f': -0.6666666666666666, 'region': 'X-'}
{'x': (1, 1, 0, 1), 'muTx': 9.0, 'N': 4.0, 'D': 11.0, 'f': -0.6030226891555273, 'region': 'X-'}
{'x': (1, 0, 1, 0), 'muTx': 8.0, 'N': 1.0, 'D': 6.0, 'f': -0.4082482904638631, 'region': 'X-'}
{'x': (0, 0, 1, 1), 'muTx': 7.0, 'N': 0.0, 'D': 8.0, 'f': 0.0, 'region': 'X+'}
{'x': (1, 1, 0, 0), 'muTx': 7.0, 'N': 0.0, 'D': 7.0, 'f': 0.0, 'region': 'X+'}
{'x': (0, 1, 0, 1), 'muTx': 6.0, 'N': 1.0, 'D': 5.0, 'f': 0.4472135954999579, 'region': 'X+'}
{'x': (1, 0, 0, 1), 'muTx': 5.0, 'N': 4.0, 'D':